In [ ]:
"""
베이스라인 초고속 확인
 - CV 없음: train 80% / val 20% 단순 분할
 - 데이터 50% 샘플링 (CatBoost 속도용)
 - iterations 100, lr=0.1
 목적: 3개 모델 상대 성능 비교, 절대 AUC 수치는 참고용
"""

import random, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

TRAIN_PATH = "../data/train.csv"
SEED       = 42
SAMPLE_FRAC = 0.5   # 데이터 샘플 비율 (CatBoost 속도용)

Q_COLS  = [f"Q{chr(ord('a')+i)}A" for i in range(20)]
QE_COLS = [f"Q{chr(ord('a')+i)}E" for i in range(20)]
TP_COLS = [f"tp{str(i).zfill(2)}" for i in range(1, 11)]
WR_COLS = [f"wr_{str(i).zfill(2)}" for i in range(1, 14)]
WF_COLS = [f"wf_{str(i).zfill(2)}" for i in range(1, 4)]
REVERSE_Q = ["QaA","QdA","QeA","QfA","QgA","QiA","QkA","QnA","QqA","QrA"]
CAT_COLS  = ["age_group","education","engnat","gender","hand","married","race","religion","urban"]

random.seed(SEED); np.random.seed(SEED)

def preprocess(df):
    d = df.copy()
    if "index" in d.columns: d.drop(columns=["index"], inplace=True)
    avail_tp = [c for c in TP_COLS if c in d.columns]
    d["tp_notapplicable_cnt"] = (d[avail_tp] == 7).sum(axis=1)
    d["tp_missing_cnt"]       = (d[avail_tp] == 0).sum(axis=1)
    for col in avail_tp:
        d[col] = d[col].replace({0: np.nan, 7: np.nan})
    for col in ["education","engnat","hand","married","urban"]:
        if col in d.columns: d[col] = d[col].replace(0, np.nan)
    for col in REVERSE_Q:
        if col in d.columns: d[col] = 6 - d[col]
    avail_q = [c for c in Q_COLS if c in d.columns]
    d["mach_score"]      = d[avail_q].mean(axis=1)
    d["q_response_std"]  = d[avail_q].std(axis=1)
    d["q_extreme_ratio"] = ((d[avail_q]==1)|(d[avail_q]==5)).sum(axis=1)/len(avail_q)
    avail_wr = [c for c in WR_COLS if c in d.columns]
    avail_wf = [c for c in WF_COLS if c in d.columns]
    d["vocab_real"]     = d[avail_wr].sum(axis=1)
    d["vocab_fake"]     = d[avail_wf].sum(axis=1)
    d["vocab_score"]    = d["vocab_real"] - d["vocab_fake"] * 2
    d["vocab_accuracy"] = d["vocab_real"] / (d["vocab_real"] + d["vocab_fake"] + 1e-6)
    avail_qe = [c for c in QE_COLS if c in d.columns]
    d["delay_root10"] = np.power(d[avail_qe].sum(axis=1).clip(lower=0), 0.1)
    d = d.drop(columns=avail_qe)
    if "familysize" in d.columns:
        d["familysize"] = np.log1p(d["familysize"].clip(lower=0))
    return d

# ── 데이터 로드 & 전처리 ─────────────────────────────────
raw = pd.read_csv(TRAIN_PATH)
raw["voted"] = (raw["voted"] == 1).astype(int)
raw = raw[raw["familysize"] <= 50].reset_index(drop=True)

feat = preprocess(raw)
y = feat["voted"].values
X = feat.drop(columns=["voted"])

# 50% 샘플링
idx = np.random.choice(len(X), int(len(X) * SAMPLE_FRAC), replace=False)
X, y = X.iloc[idx].reset_index(drop=True), y[idx]

# hold-out 분할
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=SEED)

# CatBoost용 범주형
for col in CAT_COLS:
    if col in X_tr.columns:
        X_tr[col] = X_tr[col].astype(str)
        X_va[col] = X_va[col].astype(str)
cat_idx = [i for i, c in enumerate(X_tr.columns) if c in CAT_COLS]

print(f"학습: {len(X_tr)}행 | 검증: {len(X_va)}행 | 피처: {X_tr.shape[1]}개\n")

# ── 모델 정의 ─────────────────────────────────────────────
results = {}

# LGBM
m = lgb.train(
    dict(objective="binary", metric="auc", num_leaves=31,
         learning_rate=0.1, scale_pos_weight=1.20665,
         subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
         verbose=-1, seed=SEED),
    lgb.Dataset(X_tr, label=y_tr),
    num_boost_round=100,
    valid_sets=[lgb.Dataset(X_va, label=y_va)],
    callbacks=[lgb.early_stopping(15, verbose=False), lgb.log_evaluation(9999)],
)
results["LGBM"] = roc_auc_score(y_va, m.predict(X_va, num_iteration=m.best_iteration))

# XGB
m = xgb.train(
    dict(objective="binary:logistic", eval_metric="auc",
         max_depth=4, learning_rate=0.1,
         subsample=0.8, colsample_bytree=0.8,
         scale_pos_weight=1.20665, tree_method="hist",
         verbosity=0, seed=SEED),
    xgb.DMatrix(X_tr, label=y_tr),
    num_boost_round=100,
    evals=[(xgb.DMatrix(X_va, label=y_va), "val")],
    early_stopping_rounds=15, verbose_eval=False,
)
results["XGB"] = roc_auc_score(y_va, m.predict(xgb.DMatrix(X_va),
                                iteration_range=(0, m.best_iteration+1)))

# CatBoost
m = CatBoostClassifier(
    iterations=100, learning_rate=0.1,
    depth=4, l2_leaf_reg=5,
    bootstrap_type="Bernoulli", subsample=0.8,
    scale_pos_weight=1.20665,
    loss_function="Logloss", eval_metric="AUC",
    early_stopping_rounds=15, random_seed=SEED,
    allow_writing_files=False, verbose=False,
)
m.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_idx, use_best_model=True)
results["CatBoost"] = roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])

# ── 결과 출력 ─────────────────────────────────────────────
print("=" * 35)
print(f"{'모델':<12} {'AUC':>10}")
print("-" * 35)
for name, auc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{name:<12} {auc:>10.5f}")
print("=" * 35)
print("※ 샘플 50% + hold-out 기준 → 절대 수치는 참고용")